In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Wan inversion state sequence: fixed real experiment

Run all creates 4 content/seed cases x OFF/A/B = 12 new videos, 181 frames at
320x512, 8 fps. Outputs, logs and source archive are stored under
`MyDrive/Video-WM/InversionState/inversion_state_<UTC>`.

This new candidate writes the unchanged keyed state_clock.trajectory for message
0/1: 11 four-phase states, 10 rotations. Channel 0 initial-noise coordinates are
key-permuted and padded. Even/odd sorted coordinates carry the two state axes.
Each state uses four latent slices: [1,5), [5,9), ..., [41,45), with 5120 correlated
coordinates per axis. Channels 1-15 and boundary slices 0 and 45 stay unchanged.
There are TWO message candidates, not 22 independent payload bits.

Generation uses 50 native UniPC steps. Reception reads actual saved CRF18
MP4 pixels, VAE-encodes them, then runs the same 50-step approximate flow Euler
inverse. Known original prompt/negative/CFG5 and original latent-time coordinates
are assumed. This is not unknown-crop recovery or an exact UniPC inverse.

Receiver retains raw two-axis amplitude observations for 46 slices, 11 core
windows and 2 boundaries. Public candidate ranking uses ONLY negative mean
innovation from the unchanged fixed-gain 0.5 observer, separately with update
on/off. Ties and invalid observations stay unresolved. True-message scoring is
posthoc; updated observer states never replace raw phase observations.

Report 88 marked core windows, 176 state components and 8 full trajectories;
per-mode unique true-message decisions and margins; observer on/off changes;
OFF candidate rankings and same-case OFF MP4 quality. Fixed failures remain.
Execution completion is not scientific success, observer gain, robustness or
low FPR. No automatic selection, scan, model-name GPU gate or expanded run.

Budgets unchanged from initial-noise baseline: 2400 Transformer forwards,
600 native forward steps, 600 inverse updates, 12 VAE decodes, 12 encodes,
12 MP4 saves. Stages release their models separately. Select GPU and Run all.
Only CPU/fake and static notebook checks have been run by the assistant.

Source SHA: d85894b7ab001a5a4b40782b5aa5788aa50a6e57.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = 'd85894b7ab001a5a4b40782b5aa5788aa50a6e57'
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'inversion_state_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/InversionState') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.inversion_state_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status': result['status'], 'video_denominator': result['video_denominator'], 'case_denominator': result['case_denominator'], 'fixed_calls': result['fixed_calls'], 'state_summary': result.get('state_summary'), 'actual_calls_observed': result.get('actual_calls_observed'), 'call_count_case_coverage': result.get('call_count_case_coverage'), 'cases': {k: v['status'] for k, v in result['cases'].items()}}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
